# Coalition-Graph Neural Architecture — Specialization Test

**Goal**: Do individual nodes light up for specific operation types?

Train the coalition model on compositional arithmetic, then inspect whether:
- Different nodes activate for addition vs multiplication vs parenthesized expressions
- Coalition sizes vary by input complexity
- Learned edges connect semantically related nodes

In [ ]:
# ===== Setup =====
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: ''))

if IN_COLAB:
    !pip install -q torch pyyaml matplotlib seaborn networkx scikit-learn numpy
    if not os.path.exists('Dynamic-Coalition-Network-DCN-'):
        !git clone https://github.com/tanushappapogu-max/Dynamic-Coalition-Network-DCN-.git
    os.chdir('Dynamic-Coalition-Network-DCN-')

sys.path.insert(0, '.')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ===== Generate Data =====
import yaml
from torch.utils.data import DataLoader
from src.data.arithmetic import create_datasets

with open('configs/experiment.yaml') as f:
    config = yaml.safe_load(f)

print('Generating datasets...')
datasets = create_datasets(config)
for name, ds in datasets.items():
    print(f'  {name}: {len(ds)} examples')

print('\nSample expressions:')
for i in range(8):
    expr, result = datasets['train'].data[i]
    from src.data.arithmetic import classify_expression
    print(f'  {expr} = {result}  [{classify_expression(expr)}]')

bs = config['training']['batch_size']
train_loader = DataLoader(datasets['train'], batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(datasets['val'], batch_size=bs, num_workers=2, pin_memory=True)
test_loader = DataLoader(datasets['test'], batch_size=bs, num_workers=2, pin_memory=True)
gen_loader = DataLoader(datasets['gen_test'], batch_size=bs, num_workers=2, pin_memory=True)

In [ ]:
# ===== Train Coalition Model =====
# This is the main training cell. ~1-2 hours on T4.

!python train_coalition.py

In [ ]:
# ===== Load Trained Model =====

from src.models.coalition import CoalitionModel

model = CoalitionModel(config).to(device)
checkpoint = torch.load('results/coalition/final.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded model | {model.count_parameters():,} params')
print(f'Test accuracy:  {checkpoint["test_metrics"]["accuracy"]:.4f}')
print(f'Gen accuracy:   {checkpoint["gen_metrics"]["accuracy"]:.4f}')

In [ ]:
# ===== Specialization Analysis: Which nodes light up for which tasks? =====

from src.evaluation.metrics import collect_activation_patterns
from src.evaluation.visualize import plot_activation_heatmap, compute_cluster_purity

print('Collecting activation patterns from 1000 test examples...')
patterns = collect_activation_patterns(model, datasets['test'], device, n_samples=1000)

# Count expression types
from collections import Counter
type_counts = Counter(p['expr_type'] for p in patterns)
print(f'\nExpression types in sample:')
for t, c in type_counts.most_common():
    print(f'  {t}: {c}')

# Heatmap for each layer
for layer_idx in range(config['model']['n_layers']):
    plot_activation_heatmap(
        patterns, layer_idx=layer_idx,
        save_path=f'results/coalition/heatmap_layer{layer_idx+1}.png'
    )

# Show inline
from IPython.display import Image, display
for layer_idx in range(config['model']['n_layers']):
    print(f'\n--- Layer {layer_idx+1} ---')
    display(Image(filename=f'results/coalition/heatmap_layer{layer_idx+1}.png'))

In [ ]:
# ===== Cluster Purity: Do activation clusters map to operation types? =====

for layer_idx in range(config['model']['n_layers']):
    purity = compute_cluster_purity(patterns, n_clusters=8, layer_idx=layer_idx)
    print(f'\nLayer {layer_idx+1} — Cluster Purity: {purity["purity"]:.3f}')
    if 'clusters' in purity:
        for c, info in sorted(purity['clusters'].items()):
            print(f'  Cluster {c}: {info["dominant_type"]:20s} purity={info["purity"]:.2f} size={info["size"]}')

In [ ]:
# ===== Per-Node Specialization: What does each node prefer? =====

import numpy as np

layer_idx = 0  # Change to inspect different layers
n_nodes = config['coalition']['n_nodes']
types = ['add_sub_only', 'mul_div_only', 'mixed_precedence', 'parenthesized']

print(f'\n=== Per-Node Type Preference (Layer {layer_idx+1}) ===')
print(f'{"Node":>6}  {"Avg Act":>8}  {"Preferred Type":>20}  {"Preference Score":>16}  Breakdown')
print('-' * 90)

for node_idx in range(n_nodes):
    type_acts = {t: [] for t in types}
    for p in patterns:
        if layer_idx < len(p['activations']):
            type_acts[p['expr_type']].append(p['activations'][layer_idx][node_idx])
    
    avg_by_type = {t: np.mean(acts) if acts else 0 for t, acts in type_acts.items()}
    overall_avg = np.mean([a for acts in type_acts.values() for a in acts]) if any(type_acts.values()) else 0
    preferred = max(avg_by_type, key=avg_by_type.get)
    
    # Preference score: how much more does it activate for its preferred type vs others?
    other_avg = np.mean([v for t, v in avg_by_type.items() if t != preferred]) if len(avg_by_type) > 1 else 0
    pref_score = avg_by_type[preferred] - other_avg
    
    breakdown = ' | '.join(f'{t[:6]}={v:.2f}' for t, v in avg_by_type.items())
    print(f'{node_idx:>6}  {overall_avg:>8.3f}  {preferred:>20}  {pref_score:>16.4f}  {breakdown}')

In [ ]:
# ===== Coalition Size Distribution: Do harder inputs recruit more nodes? =====

from src.evaluation.visualize import plot_coalition_size_distribution
from IPython.display import Image, display

plot_coalition_size_distribution(patterns, save_path='results/coalition/coalition_sizes.png')
display(Image(filename='results/coalition/coalition_sizes.png'))

# Size by expression type
print('\nCoalition size by expression type:')
for t in types:
    sizes = [(p['activations'][0] > 0.5).sum() for p in patterns 
             if p['expr_type'] == t and len(p['activations']) > 0]
    if sizes:
        print(f'  {t:20s}: mean={np.mean(sizes):.1f}, std={np.std(sizes):.1f}, range=[{min(sizes)}, {max(sizes)}]')

In [ ]:
# ===== Graph Structure: What edges did the model learn? =====

from src.evaluation.visualize import plot_graph_structure
from IPython.display import Image, display

# Get edge weights from first coalition layer (raw learned weights)
ew = None
for layer in model.layers:
    if hasattr(layer.ffn, 'edge_weights'):
        ew = layer.ffn.edge_weights.detach().cpu().numpy()
        break

if ew is not None:
    # Normalize to [0,1] range for visualization
    ew_viz = (ew - ew.min()) / (ew.max() - ew.min() + 1e-8)
    plot_graph_structure(patterns, ew_viz, save_path='results/coalition/graph_structure.png')
    display(Image(filename='results/coalition/graph_structure.png'))

    # Print strongest edges (by absolute value)
    print('\nTop 10 strongest learned edges:')
    n = ew.shape[0]
    edges = []
    for i in range(n):
        for j in range(n):
            if i != j:
                edges.append((i, j, ew[i, j]))
    edges.sort(key=lambda x: -abs(x[2]))
    for src, dst, w in edges[:10]:
        print(f'  Node {src} -> Node {dst}: {w:.4f}')

In [ ]:
# ===== Training Curves =====

import json
import matplotlib.pyplot as plt

with open('results/coalition/training_log.json') as f:
    log = json.load(f)

epochs = [e['epoch'] for e in log]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(epochs, [e['train_task_loss'] for e in log], label='Train')
axes[0,0].plot(epochs, [e['val_loss'] for e in log], label='Val')
axes[0,0].set_title('Task Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(epochs, [e['val_accuracy'] for e in log])
axes[0,1].set_title('Validation Accuracy'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(epochs, [e['temperature'] for e in log])
axes[0,2].set_title('Temperature'); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(epochs, [e['coalition_size_mean'] for e in log])
axes[1,0].set_title('Avg Coalition Size'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(epochs, [e['node_freq_std'] for e in log])
axes[1,1].set_title('Node Frequency Std (specialization)'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(epochs, [e.get('n_strong_edges', 0) for e in log])
axes[1,2].set_title('Strong Edges (>0.5)'); axes[1,2].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('results/coalition/training_dashboard.png', dpi=150)
plt.show()
print('\nKey question: Does node_freq_std increase over training? That means specialization is emerging.')

In [ ]:
# ===== Verdict =====

print('='*60)
print('SPECIALIZATION VERDICT')
print('='*60)

diag = checkpoint['final_diagnostics']
test_acc = checkpoint['test_metrics']['accuracy']
gen_acc = checkpoint['gen_metrics']['accuracy']

print(f'\nTest Accuracy:       {test_acc:.4f}')
print(f'Gen Accuracy:        {gen_acc:.4f}')
print(f'Coalition Size:      {diag["coalition_size_mean"]:.1f} / {config["coalition"]["n_nodes"]}')
print(f'Coalition Variance:  {diag["coalition_size_std"]:.2f}')
print(f'Node Freq Std:       {diag["node_freq_std"]:.4f}')
print(f'Strong Edges:        {diag["n_strong_edges"]}')

purity = compute_cluster_purity(patterns, n_clusters=8, layer_idx=0)
print(f'Cluster Purity:      {purity["purity"]:.3f}')

print()
score = 0
checks = []

if 3 <= diag['coalition_size_mean'] <= 12:
    checks.append('PASS: Coalition is selective (not all/no nodes)')
    score += 1
else:
    checks.append('FAIL: Coalition not selective')

if diag['coalition_size_std'] > 0.5:
    checks.append('PASS: Coalition size varies by input')
    score += 1
else:
    checks.append('FAIL: Coalition size is fixed (no input-dependent routing)')

if diag['node_freq_std'] > 0.05:
    checks.append('PASS: Nodes have different activation frequencies (differentiation)')
    score += 1
else:
    checks.append('FAIL: All nodes activate equally (no specialization)')

if purity['purity'] > 0.4:
    checks.append(f'PASS: Activation clusters map to operation types (purity={purity["purity"]:.2f})')
    score += 1
else:
    checks.append(f'FAIL: Activation patterns do not cluster by type (purity={purity["purity"]:.2f})')

if diag['n_strong_edges'] > 0:
    checks.append(f'PASS: Model learned {diag["n_strong_edges"]} strong recruitment edges')
    score += 1
else:
    checks.append('FAIL: No strong edges learned')

for c in checks:
    print(f'  {c}')

print(f'\nScore: {score}/5')
if score >= 4:
    print('RESULT: Strong evidence of node specialization. The mechanism works.')
elif score >= 2:
    print('RESULT: Partial specialization. Mechanism shows promise but needs tuning.')
else:
    print('RESULT: Weak or no specialization. Investigate routing collapse or gradient issues.')